# `StruckeC14_Sweden_v1.csv` -> species + material transformation

Continuation of `c14_dataset_tranformation.ipynb`'s steps 1-4, retargeted at the new dataset
revision `data/StruckeC14_Sweden_v1.csv`. Scope has narrowed since that notebook: this pipeline
only melts by species and material element - the measurement melt (step 5 there) is out of scope
here.

Comparing `StruckeC14_Sweden_v1.csv` against `c14_master_v08.xlsx` first (see conversation/analysis
that preceded this notebook) showed it's the same underlying 30,301 records, lightly re-exported:
- `material` vocabulary is unchanged (same 42 values, identical counts) - the existing
  `material_counts_resolved_with_ids` resolution is reused as-is, just joined case-insensitively
  since v1's text is lowercase where the resolution was built against Title Case.
- `species` vocabulary is ~97% unchanged; the ~3% that differs is punctuation normalization
  (`cf` -> `cf.`, trailing `.` added after `sp`) plus two real content changes: `naket korn`/
  `najet korn` (previously flagged uncertain as `naket?`) merged into an unambiguous `nakenkorn`,
  and a comma-less `en möjl. gran` variant. Reconciled below rather than re-running species
  resolution from scratch - the SEAD taxonomy resolution (`output/species/manual_species_resolved_with_ids_v4_6.csv`)
  is reused unchanged.

Columns are also renamed to their Swedish/SEAD-facing names: `raa_id` -> `raa_nummer`,
`site_id` -> `lamningsnummer`, `lab_id` -> `lab_nummer`, `context_id` -> `feature_name`.


In [1]:
from pathlib import Path

import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

MOD_DATASET_DIR = Path('../output/mod_dataset')
MOD_DATASET_DIR.mkdir(parents=True, exist_ok=True)

DATA_VERSION = 'v1'

def versioned(filename):
    # e.g. 'StruckeC14_Sweden_with_sead_taxonomy.csv' -> '..._v1.csv'
    stem, suffix = filename.rsplit('.', 1)
    return f'{stem}_{DATA_VERSION}.{suffix}'

def next_available_path(filename, dir_path=MOD_DATASET_DIR):
    # Never silently overwrite a previous run's output - if the versioned filename is already
    # taken, keep bumping a numeric suffix (_2, _3, ...) until a free one is found.
    versioned_name = versioned(filename)
    candidate = dir_path / versioned_name
    if not candidate.exists():
        return candidate
    stem, suffix = versioned_name.rsplit('.', 1)
    n = 2
    while (dir_path / f'{stem}_{n}.{suffix}').exists():
        n += 1
    return dir_path / f'{stem}_{n}.{suffix}'


## 1. Load `StruckeC14_Sweden_v1.csv` and rename columns


In [2]:
RENAME_MAP = {
    'raa_id': 'raa_nummer',
    'site_id': 'lamningsnummer',
    'lab_id': 'lab_nummer',
    'context_id': 'feature_name',
}

df = pd.read_csv('../data/StruckeC14_Sweden_v1.csv', encoding='utf-8')
df = df.rename(columns=RENAME_MAP)
print(f'{len(df)} rows loaded, {len(df.columns)} columns')
df.columns.tolist()


30301 rows loaded, 37 columns


['socken',
 'landskap',
 'place_name',
 'raa_nummer',
 'site_type',
 'lamningsnummer',
 'uppdragsnummer',
 'lab_nummer',
 'c14_age_bp',
 'c14_error',
 'd13C',
 'pMC_value',
 'pMC_error',
 'c14_data_status',
 'comment',
 'assessed_relevant',
 'material',
 'species',
 'feature_name',
 'context_type',
 'northing_3006',
 'easting_3006',
 'longitude',
 'latitude',
 'location_precision',
 'cal_68_min',
 'cal_68_max',
 'cal_95_min',
 'cal_95_max',
 'median_cal_year',
 'calibration_method',
 'calibration_status',
 'author',
 'publication_year',
 'title',
 'journal',
 'place_of_publication']

## 2. Split/melt `species` into one value per row

Same approach as `species_study.ipynb`/`c14_dataset_tranformation.ipynb`: lowercase, split on
`,`/`/`, strip stray digits/`?`, melt so each record contributes one row per split species value
(records with no species value at all keep a single row with `species_split = NaN`).


In [3]:
import re

df['species'] = df['species'].str.lower()

noise_pattern = re.compile(r'[0-9?]')

def clean_text(value):
    if pd.isna(value) or value == '':
        return pd.NA
    cleaned = noise_pattern.sub('', value).strip()
    return cleaned if cleaned else pd.NA

species_parts = df['species'].str.split(r'[,/]', expand=True)
species_parts.columns = [f'species_{i + 1}' for i in range(species_parts.shape[1])]
species_parts = species_parts.apply(lambda col: col.map(clean_text))
df = df.join(species_parts)

species_cols = [c for c in df.columns if c.startswith('species_')]
species_split = df[species_cols].stack().dropna().droplevel(1).rename('species_split')
melted = df.drop(columns=species_cols).join(species_split).reset_index(drop=True)
print(f'{len(df)} original rows -> {len(melted)} rows after splitting/melting species')


30301 original rows -> 30805 rows after splitting/melting species


## 3. Reconcile `species_split` tokens against the existing manual mapping

`data/species_split_counts_in_original_manual_v4.csv` (411 tokens) was built against the old
export's `species` text. Most of v1's tokens match it verbatim; where they don't, try normalizing
away the punctuation v1 added (`cf` -> `cf.`, trailing `.` after `sp`) before falling back to two
explicit overrides for the tokens that are genuinely new:

- `nakenkorn` -> `korn` (the old `naket korn`/`najet korn` rows were mapped to the uncertain
  `naket?` - v1's cleaned-up spelling is resolved directly to `korn` instead).
- `en möjl. gran` -> `gran`.

Anything left unresolved after that would need manual review before proceeding - asserted away
here since the reconciliation above accounts for every token in this revision.


In [4]:
MANUAL_MAPPING_PATH = Path('../data/species_split_counts_in_original_manual_v4.csv')
manual_map_raw = pd.read_csv(
    MANUAL_MAPPING_PATH, keep_default_na=False, na_values=[''],
)[['species_split', 'manual_species']]

def normalize_token(t):
    if pd.isna(t):
        return t
    t = t.replace('cf.', 'cf').replace(' sp.', ' sp')
    return t.rstrip('.').strip()

known_tokens = set(manual_map_raw['species_split'].dropna().astype(str))
melted_tokens = set(melted['species_split'].dropna().astype(str))
unmatched = melted_tokens - known_tokens

norm_lookup = {}
for t in known_tokens:
    norm_lookup.setdefault(normalize_token(t), t)

EXPLICIT_OVERRIDES = {
    'nakenkorn': 'korn',
    'en möjl. gran': 'gran',
}

normalization_map = {}  # v1 token -> v4 token it stands in for
for t in unmatched:
    if t in EXPLICIT_OVERRIDES:
        continue
    norm_t = normalize_token(t)
    if norm_t in norm_lookup:
        normalization_map[t] = norm_lookup[norm_t]

still_unresolved = sorted(unmatched - set(normalization_map) - set(EXPLICIT_OVERRIDES))
print(f'{len(unmatched)} tokens not present verbatim in the v4 manual mapping')
print(f'  {len(normalization_map)} resolved via punctuation normalization')
print(f'  {len(EXPLICIT_OVERRIDES)} resolved via explicit override')
print(f'  {len(still_unresolved)} still unresolved: {still_unresolved}')
assert not still_unresolved, 'unexpected unmatched species_split tokens - needs manual review'


29 tokens not present verbatim in the v4 manual mapping
  27 resolved via punctuation normalization
  2 resolved via explicit override
  0 still unresolved: []


In [5]:
def split_manual_species(value):
    if pd.isna(value):
        return [pd.NA]
    return [part.strip() for part in value.split(',')]

manual_species_split_map = pd.DataFrame([
    {'species_split': row.species_split, 'manual_species': part}
    for row in manual_map_raw.itertuples(index=False)
    for part in split_manual_species(row.manual_species)
])

override_rows = pd.DataFrame(
    [{'species_split': k, 'manual_species': v} for k, v in EXPLICIT_OVERRIDES.items()]
)
normalized_rows = pd.DataFrame([
    {'species_split': v1_token, 'manual_species': ms}
    for v1_token, v4_token in normalization_map.items()
    for ms in split_manual_species(
        manual_map_raw.loc[manual_map_raw['species_split'] == v4_token, 'manual_species'].iloc[0]
    )
])
manual_species_split_map_v1 = pd.concat(
    [manual_species_split_map, normalized_rows, override_rows], ignore_index=True
)

melted_with_manual = melted.merge(manual_species_split_map_v1, on='species_split', how='left')
print(f"{melted_with_manual['manual_species'].notna().sum()} of {len(melted_with_manual)} melted rows "
      f"matched a manual_species correction")
melted_with_manual[['species', 'species_split', 'manual_species']].head(10)


18746 of 30810 melted rows matched a manual_species correction


,species,species_split,manual_species
0,lind,lind,lind
1,NaN,NaN,NaN
2,NaN,NaN,NaN
3,NaN,NaN,NaN
4,NaN,NaN,NaN
5,NaN,NaN,NaN
6,NaN,NaN,NaN
7,NaN,NaN,NaN
8,NaN,NaN,NaN
9,NaN,NaN,NaN


## 4. Attach resolved SEAD taxonomy

Species vocabulary is unchanged from the last resolution pass, so the frozen
`output/species/manual_species_resolved_with_ids_v4_6.csv` (the latest `manually_resolved_final`
from `c14_dataset_tranformation.ipynb`) is reused directly - no DB round-trip needed.


In [6]:
resolved_species = pd.read_csv('../output/species/manual_species_resolved_with_ids_v4_6.csv')

SEAD_COLUMN_MAP = {
    'common_name_text': 'common_name',
    'resolved_species': 'sead_species',
    'resolved_genus': 'sead_genus',
    'resolved_family': 'sead_family',
    'resolved_order': 'sead_order',
    'taxon_id': 'sead_species_id',
    'resolved_genus_id': 'sead_genus_id',
    'resolved_family_id': 'sead_family_id',
    'resolved_order_id': 'sead_order_id',
    'common_name_id': 'sead_common_name_id',
    'common_name_language': 'sead_common_name_language_id',
}

resolved_for_join = resolved_species[['manual_species'] + list(SEAD_COLUMN_MAP)].rename(
    columns=SEAD_COLUMN_MAP
)

with_sead = melted_with_manual.merge(resolved_for_join, on='manual_species', how='left')
with_sead = with_sead.rename(columns={'manual_species': 'species_manually_assigned'})

print(f"{with_sead['sead_species_id'].notna().sum()} of {len(with_sead)} rows have a "
      f"sead_species_id attached")
with_sead[
    ['species', 'species_split', 'species_manually_assigned', 'common_name', 'sead_genus',
     'sead_family', 'sead_order', 'sead_species_id']
].head(10)


30810 of 30810 rows have a sead_species_id attached

,species,species_split,species_manually_assigned,common_name,sead_genus,sead_family,sead_order,sead_species_id
0,lind,lind,lind,lind,Tilia,Tiliaceae,Malvales,47020
1,NaN,NaN,NaN,NaN,Indet.,Indet.,Indet.,47014
2,NaN,NaN,NaN,NaN,Indet.,Indet.,Indet.,47014
3,NaN,NaN,NaN,NaN,Indet.,Indet.,Indet.,47014
4,NaN,NaN,NaN,NaN,Indet.,Indet.,Indet.,47014
5,NaN,NaN,NaN,NaN,Indet.,Indet.,Indet.,47014
6,NaN,NaN,NaN,NaN,Indet.,Indet.,Indet.,47014
7,NaN,NaN,NaN,NaN,Indet.,Indet.,Indet.,47014
8,NaN,NaN,NaN,NaN,Indet.,Indet.,Indet.,47014
9,NaN,NaN,NaN,NaN,Indet.,Indet.,Indet.,47014


In [7]:
C14_WITH_SEAD_PATH = next_available_path('StruckeC14_Sweden_with_sead_taxonomy.csv')
with_sead.to_csv(C14_WITH_SEAD_PATH, index=False)
print(f'Saved {len(with_sead)} rows to {C14_WITH_SEAD_PATH}')


Saved 30810 rows to ..\output\mod_dataset\StruckeC14_Sweden_with_sead_taxonomy_v1.csv


## 5. Attach resolved material elements

`material` vocabulary and counts are unchanged from the last resolution pass, so the frozen
`output/material/material_counts_resolved_with_ids_v4_3.csv` is reused directly. v1's `material`
text is lowercase where that resolution was built against Title Case originals, so the join key is
lowercased on both sides (the display column itself is left untouched).


In [8]:
material_df = pd.read_csv('../output/material/material_counts_resolved_with_ids_v4_3.csv')

material_for_join = material_df[
    ['material', 'sead_element_1', 'sead_element_1_id', 'sead_element_2', 'sead_element_2_id',
     'sead_element_3', 'sead_element_3_id', 'sead_modification_type', 'sead_modification_type_id',
     'sead_record_type_id']
].copy()
material_for_join['material_lc'] = material_for_join['material'].str.lower().str.strip()
material_for_join = material_for_join.drop(columns=['material'])

with_sead['material_lc'] = with_sead['material'].str.lower().str.strip()
with_material = with_sead.merge(material_for_join, on='material_lc', how='left').drop(columns=['material_lc'])

print(f"{with_material['sead_element_1'].notna().sum()} of {len(with_material)} rows matched a "
      f"resolved material element")


30809 of 30810 rows matched a resolved material element


## 6. Melt into one material element per row

Same "stack the parts, drop the blanks" idea as the species split: `sead_element_1/2/3` become a
single `element_name` column, one row per element. Every output row now identifies exactly one
species and one material element - no measurement melt in this pipeline.


In [9]:
element_frames = []
for i in (1, 2, 3):
    other_cols = [f'sead_element_{j}{suffix}' for j in (1, 2, 3) if j != i for suffix in ('', '_id')]
    frame = with_material.drop(columns=other_cols).rename(
        columns={f'sead_element_{i}': 'element_name', f'sead_element_{i}_id': 'sead_element_id'}
    )
    element_frames.append(frame)

final_df = pd.concat(element_frames, ignore_index=True)
final_df = final_df[final_df['element_name'].notna()].reset_index(drop=True)

print(f'{len(with_material)} rows before the element melt -> {len(final_df)} rows after '
      f'(one row per species per material element)')
final_df[
    ['species_manually_assigned', 'material', 'element_name', 'sead_element_id',
     'sead_modification_type', 'sead_modification_type_id', 'sead_record_type_id']
].head(10)


30810 rows before the element melt -> 30819 rows after (one row per species per material element)


,species_manually_assigned,material,element_name,sead_element_id,sead_modification_type,sead_modification_type_id,sead_record_type_id
0,lind,träkol,Charcoal,50.0,NaN,NaN,9.0
1,NaN,träkol,Charcoal,50.0,NaN,NaN,9.0
2,NaN,träkol,Charcoal,50.0,NaN,NaN,9.0
3,NaN,träkol,Charcoal,50.0,NaN,NaN,9.0
4,NaN,träkol,Charcoal,50.0,NaN,NaN,9.0
5,NaN,träkol,Charcoal,50.0,NaN,NaN,9.0
6,NaN,träkol,Charcoal,50.0,NaN,NaN,9.0
7,NaN,träkol,Charcoal,50.0,NaN,NaN,9.0
8,NaN,träkol,Charcoal,50.0,NaN,NaN,9.0
9,NaN,träkol,Charcoal,50.0,NaN,NaN,9.0


In [10]:
C14_FINAL_PATH = next_available_path('StruckeC14_Sweden_with_sead_taxonomy_and_material.csv')
final_df.to_csv(C14_FINAL_PATH, index=False)
print(f'Saved {len(final_df)} rows to {C14_FINAL_PATH}')


Saved 30819 rows to ..\output\mod_dataset\StruckeC14_Sweden_with_sead_taxonomy_and_material_v1.csv
